# Representative toy image: pixels, annotated ions and spectra

Describes the small image used by all toy models: which pixels and ions it contains, how the display regions are defined, and what its spectra look like. It answers whether the toy image carries distinct, spatially organized spectral classes that a four-dimensional latent space can separate.

## Introduction

The toy campaign (`kidney-representative-toy-circle`) trains cumulative objectives on one small kidney image with latent width $L = 3$, so that the whole latent space is $S^1$ and can be shown directly. Its figures are used in the conceptual presentation (`assets/documents/prestentations/segmentation_model_full_conceptual`).

### Assumptions

- One model per objective (one repetition). Differences between variants are single realizations, not
  estimates of an expected effect; the toy is illustrative, not a benchmark.
- All variants share the data split, the initialization seed and the training seed, so the only
  difference is the objective.
- Architecture, optimizer, loss parameters and loss weights follow the `real_only` schedule of the
  `20_09_26_metaspace_base_pretrain` campaign (without label-weighted contrastive negatives), including
  its 10 epochs; only `latent_dim` is smaller.
- Ten epochs on the training part of 16 230 pixels are about 2000 optimizer steps; the
  auxiliary terms (weights 0.2, 0.001, 0.002) contribute little to the objective, so differences between
  variants are expected to be small.
- Display categories are built from five selected ions (three spatially disjoint, one overlapping one of
  them, one present almost everywhere) by the rules in `analysis_settings.yaml`; names list the ions.

### Notation

| symbol | meaning |
|---|---|
| $x \in \Delta^{M-1}$ | TIC-normalized binned spectrum, $M = 1364$ bins of $0.55$ m/z on $[200, 950]$ |
| $a \in \mathbb{R}^{L}$ | encoder output before the bottleneck `LayerNorm`, $L = 3$ |
| $u = (a - \mu_a \mathbf 1)/\sigma_a$ | canonical latent, $\mathbf 1^\top u = 0$, $\lVert u \rVert = \sqrt L$ |
| $z = \gamma \odot u + \beta$ | model latent (affine `LayerNorm` output) |
| $s = Q^\top u / \lVert Q^\top u \rVert \in S^1$ | latent coordinates; $Q \in \mathbb{R}^{3 \times 2}$ fixed orthonormal basis of $\mathbf 1^\perp$ |
| $\mathcal M_0, \dots, \mathcal M_4$ | objectives: reconstruction; + head; + head + contrastive; + head + contractive; + head + both |
| $\angle(u, u')$ | angle between canonical latents, in degrees |

## Configuration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from msi_autoencoder_wrapper.analysis.autoencoder.experiments import representative_toy_precompute as toy
from msi_autoencoder_wrapper.visualization import representative_toy as viz

SETTINGS = toy.load_settings(Path("analysis_settings.yaml"))
FIGURES = Path(SETTINGS["repository_root"]) / SETTINGS["figures"]["presentation_directory"]
LABELS = SETTINGS["campaign"]["labels"]
SHOWN = SETTINGS["campaign"]["presentation"]  # variants used on the slides
PREFIX = SETTINGS["figures"]["prefix"]
REGION_COLORS = {rule["name"]: rule["color"] for rule in SETTINGS["regions"]}

classes = toy.load_table(SETTINGS, "dataset", "classes")
pixels = toy.load_table(SETTINGS, "dataset", "pixels")
pixels["region"] = toy.assign_regions(pixels, classes, SETTINGS["regions"])

## Producing the tables this notebook reads

The tables are written by the toy precomputation module; the command below is generated by that module.

In [ ]:
print(toy.run_command(SETTINGS, "dataset"))

## Pixel layout, display regions and ion annotations

### Methodology

#### Theoretical

Every pixel $p$ of the toy image has a binary annotation vector $y_p \in \{0,1\}^{C}$, $C = 21$, with $y_{pc} = 1$ when METASPACE reports ion $c$ in that pixel. Display regions are the first matching marker rule (see Assumptions).

#### Implementation

The image joins two sections of one series side by side, all pixels (16 230): METASPACE datasets `2024-02-20_01h57m32s` (d28-2017-2) and `2024-02-20_01h54m41s` (d28-2006-2), restricted to peaks in $[200, 950]$ m/z; the ions are the union of both annotation sets (7 shared), and an ion not annotated in one section is unlabelled there (`build_representative_dataset.py`). Labels are read through the library's annotation reader and dataset, exactly as during training.

#### Figure descriptions

Left: toy image (both sections), one cell per pixel, colour = display category built from the five selected ions. Right: one panel per selected ion (title = m/z); coloured = ion annotated in the pixel, grey = not annotated.

In [ ]:
display(pixels.region.value_counts().rename("pixels").to_frame())
display(pixels.groupby("split").size().rename("pixels").to_frame())

In [ ]:
with viz.presentation_style():
    figure = viz.plot_region_and_ions(pixels, classes, REGION_COLORS, selected_ions=SETTINGS["selected_ions"])
    viz.save_figure(figure, FIGURES, PREFIX + "regions_ions")
    plt.show()

### Remarks

### Notes

## Mean spectrum per display region

### Methodology

#### Theoretical

The mean TIC-normalized spectrum $\bar x_R = |R|^{-1} \sum_{p \in R} x_p$ of each region $R$ shows which bins distinguish the regions.

#### Implementation

Spectra are the binned, TIC-normalized model inputs decoded by the training dataset (`spectra.csv`, long format of non-zero bins).

#### Figure descriptions

x = m/z (bin centre), y = mean normalized intensity; one line per region (colour = region); vertical dotted lines mark the annotated ion m/z.

In [ ]:
spectra = toy.load_table(SETTINGS, "dataset", "spectra").merge(pixels[["source_id", "region"]], on="source_id")
bins = toy.load_metadata(SETTINGS, "dataset")["bin_count"]
counts = pixels.region.value_counts()
mean = spectra.groupby(["region", "mz"]).intensity.sum().div(counts, level="region").rename("mean").reset_index()
with viz.presentation_style():
    figure, ax = plt.subplots(figsize=(14, 4.8))
    for region, colour in REGION_COLORS.items():
        frame = mean[mean.region == region]
        if not frame.empty:
            ax.vlines(frame.mz, 0, frame["mean"], color=colour, lw=1.4, alpha=0.85, label=region)
    for mz in classes.mz:
        ax.axvline(mz, color=viz.MUTED, ls=":", lw=0.8)
    ax.set_xlabel("m/z")
    ax.set_ylabel("mean intensity (TIC = 1)")
    ax.set_title(f"toy spectra: {bins} bins of 0.55 m/z")
    ax.legend(frameon=False, ncol=3)
    viz.save_figure(figure, FIGURES, PREFIX + "region_spectra")
    plt.show()

### Remarks

### Notes

## Results / Summary

### LLM

### Person